# Лабораторная работа 2. Тесты класса Cache

In [1]:
import time
import threading
from cache import Cache

## set / get / delete

In [2]:
cache = Cache()
cache.set('a', 1)
cache.set('b', 'hello')
print('a =', cache.get('a'))
print('b =', cache.get('b'))
print('missing =', cache.get('missing'))
print('deleted a:', cache.delete('a'))
print('a after delete =', cache.get('a'))

a = 1
b = hello
missing = None
deleted a: 1
a after delete = None


## Ключ не может быть None

In [3]:
try:
    cache.set(None, 1)
except ValueError as e:
    print('OK, set(None) ->', e)
try:
    cache.get(None)
except ValueError as e:
    print('OK, get(None) ->', e)
try:
    cache.delete(None)
except ValueError as e:
    print('OK, delete(None) ->', e)

OK, set(None) -> key must not be None
OK, get(None) -> key must not be None
OK, delete(None) -> key must not be None


## TTL: ключ истекает по таймеру (фоновый поток)

In [4]:
cache.set('temp', 42, ttl=1.0)
print('сразу:', cache.get('temp'))
time.sleep(0.5)
print('через 0.5с:', cache.get('temp'))
time.sleep(0.8)
print('через 1.3с:', cache.get('temp'))
# проверим, что фоновый поток сам подчистил данные из внутреннего хранилища
print('внутри _store:', 'temp' in cache._store)

сразу: 42
через 0.5с: 42
через 1.3с: None
внутри _store: False


## compute_if_present

In [5]:
cache.set('counter', 10)
result = cache.compute_if_present('counter', lambda v: v + 5)
print('новое значение:', result)
print('из кэша:', cache.get('counter'))
# для отсутствующего ключа возвращает None и функция не вызывается
result2 = cache.compute_if_present('nope', lambda v: v + 1)
print('для отсутствующего:', result2)

новое значение: 15
из кэша: 15
для отсутствующего: None


## Потокобезопасность compute_if_present
3 потока × 1000 итераций инкремента. Ожидаем counter = 3000.

In [6]:
cache.set('counter', 0)

def worker():
    for _ in range(1000):
        cache.compute_if_present('counter', lambda v: v + 1)

threads = [threading.Thread(target=worker) for _ in range(3)]
for t in threads:
    t.start()
for t in threads:
    t.join()

final = cache.get('counter')
print('итоговое значение:', final)
assert final == 3000, f'ожидали 3000, получили {final}'

итоговое значение: 3000


In [7]:
cache.close()